<a href="https://colab.research.google.com/github/piaoruilin/hallucination-medical-vlm/blob/main/inject_hallucinations_v2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scale up: inject hallucinations with Qwen (v2)
Fixes from the first run: minimal-edit prompt, labels computed by **diffing** the hallucinated sentence against its clean twin (Qwen's own word list is kept only as a cross-check), error type chosen to fit the answer, failures logged with reasons, image path saved, a train/val/test split by question, and **resume after a disconnect** (rerun the notebook and it skips finished rows).

Output: `halluc_v2.jsonl` (two lines per kept pair) and `halluc_v2_failed.jsonl`. Needs a GPU.

v2.1: the error **type is inferred from what actually changed** (the requested type is kept as `type_requested`), non-answers like "No information is given" are dropped, and imaging-modality swaps (CT to MRI) are rejected by default.

In [1]:
!pip install -q "bitsandbytes>=0.46.1" accelerate transformers tqdm
# If this installs something new: Runtime > Restart session once, then skip this cell.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.7 MB/s eta 0:00:00


In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
from google.colab import drive
drive.mount("/content/drive")

GPU available: True
Mounted at /content/drive


## Settings

In [3]:
REPO = "xmcmic/PMC-VQA"
IMG_COL, Q_COL = "Figure_path", "Question"

START_ROW = 100          # rows 0-99 were used in the first run; skipping them keeps the old 77 pairs separate
N_NEW = 500              # rows to process this run. Test with 500, then raise it
BATCH = 8                # prompts generated at once (lower to 4 if you run out of GPU memory)
EXCLUDE_NON_CT = True    # drop rows whose question/answer mention X-ray, MRI, PET, microscopy, ...
MAX_CHANGED = 6          # reject a hallucinated sentence if more than this many words differ from the clean one
REJECT_MODALITY_SWAPS = True   # drop pairs where the imaging modality was swapped (CT -> MRI). False keeps them as type wrong_modality
SEED = 0

OUT  = "/content/drive/MyDrive/halluc_v2.jsonl"
FAIL = "/content/drive/MyDrive/halluc_v2_failed.jsonl"
model_id = "Qwen/Qwen2.5-7B-Instruct"   # same Qwen as before

## Load and filter the data

In [4]:
import os, re, json, random, numpy as np, pandas as pd
from collections import Counter
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

src = pd.read_csv(hf_hub_download(REPO, "train_2.csv", repo_type="dataset"))
A_RAW = next(c for c in src.columns if c.lower() in ("answer", "anwser"))
choice_col = {c.lower().replace(" ", "").replace("_", ""): c for c in src.columns}

def get_answer(row):
    a = str(row[A_RAW]).strip()
    if re.fullmatch(r"[A-Da-d]", a):                       # a bare letter: use the option's text
        col = choice_col.get("choice" + a.lower())
        if col and pd.notna(row[col]):
            a = str(row[col]).strip()
    return re.sub(r"^[A-D][\.\:\)]\s*", "", a)

src["ans"] = src.apply(get_answer, axis=1)

# SAME base filter as the first run, so `row` numbers match the old file
kw = "chest|lung|thora|pulmon|pleur|mediastin"
df = src[src[Q_COL].astype(str).str.lower().str.contains(kw)].reset_index(drop=True)

NONCT = (r"x-?ray|radiograph|\bmri\b|magnetic resonance|\bpet\b|positron|ultrasound|sonograph|echocardio|"
         r"microscop|h&e|stain|histolog|intraoperative|angiograph|fluoroscop|photograph")
txt = (df[Q_COL].astype(str) + " " + df["ans"].astype(str)).str.lower()
ok = ~txt.str.contains(NONCT) if EXCLUDE_NON_CT else pd.Series(True, index=df.index)

# drop rows whose "answer" is not a real answer (there is nothing to hallucinate about)
NONANSWER = r"no information|not (?:mentioned|given|provided|specified|stated|available)|unknown|cannot be determined|not applicable|\bn/?a\b|none of the above"
ok = ok & ~df["ans"].astype(str).str.lower().str.contains(NONANSWER)

# drop short answers that are just an imaging modality ("CT scan"): the only possible error is a modality swap
short = df["ans"].astype(str).str.split().str.len() <= 4
mod_ans = df["ans"].astype(str).str.contains(r"\b(?:ct|mri|pet|x-?rays?|ultrasound|tomography)\b", case=False, regex=True)
ok = ok & ~(short & mod_ans)

cand = df.index[(df.index >= START_ROW) & ok]
order = np.random.default_rng(SEED).permutation(cand.values)[:N_NEW]
print(len(df), "chest-keyword rows |", int(ok.sum()), "after the non-CT exclusion |", len(cand), "candidates |", len(order), "selected")

train_2.csv: reconstructing file:   0%|          |  0.00B / 56.7MB            

train_2.csv: downloading bytes:           |  0.00B            

3377 chest-keyword rows | 2308 after the non-CT exclusion | 2244 candidates | 500 selected


## Load Qwen (4-bit)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(model_id)
tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
llm.eval()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

## Prompt, validation and labelling

In [8]:
SYSTEM = ("You help build a research dataset for detecting hallucinations in medical image question answering. "
          "You output only valid JSON.")

TYPE_INSTR = {
    "wrong_organ": "replace ONE organ or anatomical structure word with a different, incorrect one",
    "wrong_side": "flip ONE left/right or position word (upper/lower, anterior/posterior) to its opposite",
    "invented_finding": "insert ONE short phrase (3 to 6 words) describing a finding that the correct answer does not support",
    "wrong_attribute": "replace ONE size, severity, count or descriptive word with an incorrect one",
}
# my mapping onto HalLoc's categories (check it against the paper before relying on it)
HALLOC_TYPE = {"wrong_organ": "object", "invented_finding": "object", "wrong_side": "attribute", "wrong_attribute": "attribute",
               "wrong_modality": "other", "wrong_finding": "object", "other": "other"}

SIDE  = re.compile(r"\b(left|right|upper|lower|middle|anterior|posterior|superior|inferior)\b", re.I)
ORGAN = re.compile(r"\b(lungs?|liver|hearts?|kidneys?|spleen|pleura\w*|aort\w*|pulmonary|lobes?|mediastin\w*|esophag\w*|trache\w*|bronch\w*|atrium|ventric\w*|vein|artery|nodes?)\b", re.I)
ATTR  = re.compile(r"\b(mild|moderate|severe|small|large|enlarged|thick\w*|diffuse|focal|multiple|single|mixed|high-grade|low-grade|\d+)\b", re.I)

rng = random.Random(SEED)
type_counts = Counter()

def choose_type(a):
    options = ["invented_finding"]
    if SIDE.search(a): options.append("wrong_side")
    if ORGAN.search(a): options.append("wrong_organ")
    if ATTR.search(a): options.append("wrong_attribute")
    low = min(type_counts[t] for t in options)          # balance by attempts, so a failing type is not retried forever
    t = rng.choice([x for x in options if type_counts[x] == low])
    type_counts[t] += 1
    return t

def build_prompt(q, a, t):
    return (
        "Steps:\n"
        "1. correct_sentence: write ONE short sentence that answers the question with the correct answer. "
        "Use only information from the question and the correct answer. Do not add facts, dates, imaging modalities or findings.\n"
        "2. hallucinated_sentence: copy correct_sentence exactly, then " + TYPE_INSTR[t] + ". "
        "Change as few words as possible. Every other word must stay identical and in the same order.\n"
        "3. changed_words: list the words in hallucinated_sentence that are new or different from correct_sentence.\n\n"
        "Example (error type wrong_side):\n"
        "Question: Which lobe contains the lesion?\nCorrect answer: Right upper lobe\n"
        '{"correct_sentence": "The lesion is in the right upper lobe.", "hallucinated_sentence": "The lesion is in the left upper lobe.", "changed_words": ["left"]}\n\n'
        "Now the real case (error type " + t + "):\n"
        "Question: " + q + "\nCorrect answer: " + a + "\n"
        "Return only JSON with the keys correct_sentence, hallucinated_sentence, changed_words.")

def generate_batch(prompts):
    texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    inputs = tok(texts, return_tensors="pt", padding=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=220, do_sample=False, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]

def norm_word(w):
    return re.sub(r"[^a-z0-9]", "", str(w).lower())

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

import difflib
def diff_ops(h_tokens, c_tokens):
    # returns (0/1 labels over the hallucinated words, clean words removed or replaced, number of separate edit regions)
    a = [norm_word(t) for t in c_tokens]
    b = [norm_word(t) for t in h_tokens]
    lab = [0] * len(h_tokens)
    removed = []
    n_edits = 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        if tag != "equal":
            n_edits += 1
        if tag in ("replace", "insert"):
            for j in range(j1, j2):
                lab[j] = 1
        if tag in ("replace", "delete"):
            removed += c_tokens[i1:i2]
    return lab, removed, n_edits

MODALITY = re.compile(r"\b(ct|mri|mr|pet|x-?rays?|radiograph\w*|ultrasound|echo\w*|angiogra\w*|tomograph\w*)\b", re.I)

def infer_type(added, removed):
    # what Qwen actually did, judged from the words that changed (not from what it was asked to do)
    a, r = " ".join(added), " ".join(removed)
    if MODALITY.search(a) or MODALITY.search(r): return "wrong_modality"
    if not removed: return "invented_finding"                      # words were only inserted
    if SIDE.search(a) and SIDE.search(r): return "wrong_side"
    if ORGAN.search(a) and ORGAN.search(r): return "wrong_organ"
    if ATTR.search(a) or ATTR.search(r): return "wrong_attribute"
    return "wrong_finding"                                         # something else was swapped for something else (e.g. a diagnosis)

def jacc(a, b):
    A = {i for i, l in enumerate(a) if l}
    B = {i for i, l in enumerate(b) if l}
    return len(A & B) / max(1, len(A | B))

FRAMING = {"image", "images", "figure", "shows", "show", "shown", "this", "that", "there", "with", "from", "which",
           "where", "located", "visible", "seen", "present", "picture", "displays", "indicates", "appears", "answer",
           "question", "refers", "refer", "scan", "scans"}

def validate(o, q, a):
    try:
        cs, hs = o["correct_sentence"].strip(), o["hallucinated_sentence"].strip()
    except Exception:
        return None, "missing_key"
    if not cs or not hs:
        return None, "empty"
    if cs == hs:
        return None, "identical"
    ct, ht = cs.split(), hs.split()
    lab, removed, n_edits = diff_ops(ht, ct)
    n = sum(lab)
    if n == 0:
        return None, "no_change"
    if n_edits > 1:
        return None, "multiple_edits"       # the sentence was restructured, not minimally edited
    if n > MAX_CHANGED or n > 0.6 * len(ht):
        return None, "too_many_changed"
    if len(ht) - len(ct) > 8:
        return None, "length_drift"
    base = {norm_word(w) for w in (q + " " + a).split()}
    if [m for m in MODALITY.findall(cs) if norm_word(m) not in base]:
        return None, "modality_added"        # the clean sentence invented an imaging modality
    novel = [w for w in ct if (len(norm_word(w)) >= 4 or any(c.isdigit() for c in w))
             and norm_word(w) not in base and norm_word(w) not in FRAMING]
    if len(novel) > 4:
        return None, "novel_clean"
    return (cs, hs, ct, ht, lab, novel, removed), "ok"

def make_records(idx, raw, q, a, t):
    o = parse_json(raw)
    if not isinstance(o, dict):
        return None, "no_json"
    res, reason = validate(o, q, a)
    if res is None:
        return None, reason
    cs, hs, ct, ht, lab, novel, removed = res
    t_inf = infer_type([w for w, l in zip(ht, lab) if l], removed)
    if t_inf == "wrong_modality" and REJECT_MODALITY_SWAPS:
        return None, "modality_swap"
    cw = o.get("changed_words", [])
    if isinstance(cw, str):
        cw = [cw]
    S = {norm_word(x) for w in cw for x in str(w).split()} - {""}
    j = jacc(lab, [int(norm_word(w) in S) for w in ht])
    u = (int(idx) * 7919) % 100
    base = {"row": int(idx), "question": q, "gt_answer": a, "type": t_inf, "type_requested": t, "halloc_type": HALLOC_TYPE[t_inf],
            "image": str(df.loc[idx, IMG_COL]), "split": "train" if u < 70 else ("val" if u < 80 else "test"),
            "jacc": round(j, 3), "agree": bool(j >= 0.5), "novel_clean": len(novel)}
    return [{**base, "answer": hs, "tokens": ht, "labels": lab, "hallucinated": 1},
            {**base, "answer": cs, "tokens": ct, "labels": [0] * len(ct), "hallucinated": 0}], "ok"

## Quick test on one batch (nothing is saved). Read these before the full run.

In [9]:
meta = []
for idx in [int(i) for i in order[:BATCH]]:
    q, a = str(df.loc[idx, Q_COL]), str(df.loc[idx, "ans"])
    meta.append((idx, q, a, choose_type(a)))
raws = generate_batch([build_prompt(q, a, t) for _, q, a, t in meta])
for (idx, q, a, t), raw in zip(meta, raws):
    recs, reason = make_records(idx, raw, q, a, t)
    print("[%s] asked %s | original: %s | %s" % (idx, t, a, reason))
    if recs:
        h = recs[0]
        print("   TYPE GOT: ", h["type"], "(asked %s)" % h["type_requested"])
        print("   CLEAN:   ", recs[1]["answer"])
        print("   HALLUC:  ", h["answer"])
        print("   LABELLED:", [w for w, l in zip(h["tokens"], h["labels"]) if l], "| agree with Qwen's list:", h["agree"])
    else:
        print("   raw:", raw[:200])

[399] asked invented_finding | original: Disruption of branching morphogenesis | multiple_edits
   raw: ```json
{"correct_sentence": "Disruption of branching morphogenesis is the observed abnormality in lung development.", "hallucinated_sentence": "Calcification is present in addition to disruption of b
[2429] asked wrong_organ | original: Right upper lung zone | ok
   TYPE GOT:  wrong_organ (asked wrong_organ)
   CLEAN:    The cystic mass is located in the right upper lung zone.
   HALLUC:   The cystic mass is located in the right upper kidney zone.
   LABELLED: ['kidney'] | agree with Qwen's list: True
[946] asked invented_finding | original: Pulmonary Langerhans cell histiocytosis | ok
   TYPE GOT:  wrong_finding (asked invented_finding)
   CLEAN:    Pulmonary Langerhans cell histiocytosis was found in the transthoracic lung biopsy.
   HALLUC:   Granulomatous inflammation was found in the transthoracic lung biopsy.
   LABELLED: ['Granulomatous', 'inflammation'] | agree with Qwen's l

In [10]:
print(df.loc[1678, Q_COL])
print(df.loc[1760, Q_COL])

 How many pulmonary nodules are visible in the CT scan? 
 What is the location of the lung nodule seen on the computed tomography? 


## Full run (resume-safe)
Writes after every batch. If the session dies, rerun the notebook from the top and this cell continues where it stopped.

In [11]:
type_counts.clear()          # forget the attempts made in the quick-test cell
done = set()
for path in (OUT, FAIL):
    if os.path.exists(path):
        for line in open(path):
            try:
                r = json.loads(line)
            except Exception:
                continue
            done.add(r["row"])
            if path == FAIL or r.get("hallucinated") == 1:
                type_counts[r.get("type_requested", r["type"])] += 1

todo = [int(i) for i in order if int(i) not in done]
print(len(todo), "rows to process,", len(done), "already done")

kept = failed = 0
with open(OUT, "a") as fo, open(FAIL, "a") as ff:
    for s in tqdm(range(0, len(todo), BATCH)):
        meta = []
        for idx in todo[s:s + BATCH]:
            q, a = str(df.loc[idx, Q_COL]), str(df.loc[idx, "ans"])
            meta.append((idx, q, a, choose_type(a)))
        raws = generate_batch([build_prompt(q, a, t) for _, q, a, t in meta])
        for (idx, q, a, t), raw in zip(meta, raws):
            recs, reason = make_records(idx, raw, q, a, t)
            if recs is None:
                failed += 1
                ff.write(json.dumps({"row": idx, "type": t, "reason": reason, "raw": raw}) + "\n")
            else:
                kept += 1
                for r in recs:
                    fo.write(json.dumps(r) + "\n")
        fo.flush(); ff.flush()
print("this run: kept", kept, "| failed", failed)

500 rows to process, 0 already done


  0%|          | 0/63 [00:00<?, ?it/s]

this run: kept 480 | failed 20


## Summary of everything saved so far

In [12]:
assert os.path.exists(OUT) and os.path.getsize(OUT) > 0, "nothing saved yet: run the full-run cell first"
v = pd.read_json(OUT, lines=True)
hv = v[v["hallucinated"] == 1]
print(len(hv), "pairs |", len(v), "answers")
print("by type:     ", hv["type"].value_counts().to_dict())
print("by split:    ", hv["split"].value_counts().to_dict())
print("type match:  ", round((hv["type"] == hv["type_requested"]).mean(), 3), "of pairs: the type Qwen produced equals the type asked for")
print(pd.crosstab(hv["type_requested"], hv["type"]))
print("agreement:   ", round(hv["agree"].mean(), 3), "of pairs have diff labels matching Qwen's own list")
print("words changed (median):", int(hv["labels"].apply(sum).median()))
if os.path.exists(FAIL) and os.path.getsize(FAIL) > 0:
    f = pd.read_json(FAIL, lines=True)
    print("failures:    ", f["reason"].value_counts().to_dict())

480 pairs | 960 answers
by type:      {'wrong_side': 215, 'wrong_finding': 146, 'invented_finding': 54, 'wrong_organ': 54, 'wrong_attribute': 11}
by split:     {'train': 344, 'test': 89, 'val': 47}
type match:   0.554 of pairs: the type Qwen produced equals the type asked for
type              invented_finding  wrong_attribute  wrong_finding  \
type_requested                                                       
invented_finding                54                0             90   
wrong_attribute                  0               11              7   
wrong_organ                      0                0             47   
wrong_side                       0                0              2   

type              wrong_organ  wrong_side  
type_requested                             
invented_finding            1           4  
wrong_attribute             1           5  
wrong_organ                52          57  
wrong_side                  0         149  
agreement:    0.977 of pairs have diff